# Transit OOH Location Value Analysis — PoC

**목적:** 라스트마일 배송차량 GPS 궤적에서 방문빈도·체류시간·재방문 특성을 추출하고, 외부 공간정보와 결합하여 **상대적 Transit OOH 위치가치**를 산출·시각화하는 분석 파이프라인의 구현 가능성을 확인한다.

> **PoC 범위**
> - 실제 배송차량 GPS 및 실측 광고효과 데이터 확보 전 단계이다.
> - 차량 궤적과 공간정보는 **시나리오 기반 모의 데이터**를 사용한다.
> - 본 노트북은 광고효과나 추가 설명력을 최종적으로 입증하는 것이 아니라, **데이터 처리 → 특징 추출 → 점수화 → GIS 시각화**의 end-to-end 구현을 검증한다.
> - 실측 데이터 확보 후 정적 공간지표 대비 궤적 기반 지표의 **증분 설명력(예: ΔR²)** 을 검증하는 것을 후속 단계로 둔다.

## 1. Environment & Parameters
재현 가능한 PoC를 위해 주요 임계값과 가중치를 한 곳에서 관리한다. 현재 가중치는 **초기 가정값(initial parameters)** 이며, 아래 민감도 분석에서 결과의 안정성을 함께 확인한다.

In [1]:
import math
import numpy as np
import pandas as pd
import folium
import branca.colormap as cm
from branca.element import Element
from IPython.display import display

np.random.seed(42)

# 분석 단위 및 정차 탐지 파라미터
GRID_SIZE = 0.002          # 약 200m 수준의 PoC 격자
STOP_SPEED_KMH = 1.0       # 정차 후보 속도 임계값
MIN_DWELL_MIN = 10         # 최소 체류시간
SAMPLE_INTERVAL_MIN = 5

# Mobility Score 초기 가중치
W_VISIT = 0.4
W_DWELL = 0.4
W_REPEAT = 0.2

# Proposed Location Value 초기 가중치
ALPHA_MOBILITY = 0.6
BETA_SPATIAL = 0.4

print("환경 설정 완료")

환경 설정 완료


## 2. Scenario-based GPS Data Generation
실측 배송차량 GPS 확보 전 단계이므로, 반복 방문·이동·체류 패턴이 포함된 **모의 배송 시나리오**를 생성한다. 이 데이터는 결과값의 일반화를 위한 것이 아니라 파이프라인 동작 검증용이다.

In [2]:
# 서울 도심을 기준으로 한 모의 배송지
locations = {
    "A": (37.5665, 126.9780),
    "B": (37.5700, 126.9820),
    "C": (37.5625, 126.9840),
    "D": (37.5680, 126.9720),
}

rows = []

def add_point(vehicle_id, timestamp, lat, lon, scenario_state):
    rows.append({
        "vehicle_id": vehicle_id,
        "timestamp": timestamp,
        "latitude": lat,
        "longitude": lon,
        "scenario_state": scenario_state,  # 생성 검증용 라벨, 분석에는 사용하지 않음
    })

def add_stop(vehicle_id, start_time, location_name, duration_min):
    lat, lon = locations[location_name]
    for minute in range(0, duration_min, SAMPLE_INTERVAL_MIN):
        add_point(
            vehicle_id,
            start_time + pd.Timedelta(minutes=minute),
            lat + np.random.normal(0, 0.00002),
            lon + np.random.normal(0, 0.00002),
            "STOP",
        )

def add_move(vehicle_id, start_time, start_loc, end_loc, duration_min):
    start_lat, start_lon = locations[start_loc]
    end_lat, end_lon = locations[end_loc]
    steps = max(2, duration_min // SAMPLE_INTERVAL_MIN)

    for i in range(steps):
        ratio = i / (steps - 1)
        lat = start_lat + (end_lat - start_lat) * ratio
        lon = start_lon + (end_lon - start_lon) * ratio
        add_point(
            vehicle_id,
            start_time + pd.Timedelta(minutes=SAMPLE_INTERVAL_MIN * i),
            lat + np.random.normal(0, 0.00002),
            lon + np.random.normal(0, 0.00002),
            "MOVE",
        )

In [ ]:
# V01: A → B → A → C
t = pd.Timestamp("2026-09-25 09:00")
add_stop("V01", t, "A", 15); t += pd.Timedelta(minutes=15)
add_move("V01", t, "A", "B", 20); t += pd.Timedelta(minutes=20)
add_stop("V01", t, "B", 20); t += pd.Timedelta(minutes=20)
add_move("V01", t, "B", "A", 20); t += pd.Timedelta(minutes=20)
add_stop("V01", t, "A", 20); t += pd.Timedelta(minutes=20)
add_move("V01", t, "A", "C", 20); t += pd.Timedelta(minutes=20)
add_stop("V01", t, "C", 10)

# V02: B → C → B → D
t = pd.Timestamp("2026-09-25 09:10")
add_stop("V02", t, "B", 15); t += pd.Timedelta(minutes=15)
add_move("V02", t, "B", "C", 20); t += pd.Timedelta(minutes=20)
add_stop("V02", t, "C", 20); t += pd.Timedelta(minutes=20)
add_move("V02", t, "C", "B", 20); t += pd.Timedelta(minutes=20)
add_stop("V02", t, "B", 15); t += pd.Timedelta(minutes=15)
add_move("V02", t, "B", "D", 20); t += pd.Timedelta(minutes=20)
add_stop("V02", t, "D", 10)

# V03: A → D → A
t = pd.Timestamp("2026-09-25 09:20")
add_stop("V03", t, "A", 10); t += pd.Timedelta(minutes=10)
add_move("V03", t, "A", "D", 20); t += pd.Timedelta(minutes=20)
add_stop("V03", t, "D", 15); t += pd.Timedelta(minutes=15)
add_move("V03", t, "D", "A", 20); t += pd.Timedelta(minutes=20)
add_stop("V03", t, "A", 15)

gps = pd.DataFrame(rows).sort_values(["vehicle_id", "timestamp"]).reset_index(drop=True)
print(f"차량 수: {gps['vehicle_id'].nunique()} | GPS 기록 수: {len(gps)}")
display(gps.head(10))

차량 수: 3 | GPS 기록 수: 65


,vehicle_id,timestamp,latitude,longitude,scenario_state
0,V01,2026-09-25 09:00:00,37.566510,126.977997,STOP
1,V01,2026-09-25 09:05:00,37.566513,126.978030,STOP
2,V01,2026-09-25 09:10:00,37.566495,126.977995,STOP
3,V01,2026-09-25 09:15:00,37.566532,126.978015,MOVE
4,V01,2026-09-25 09:20:00,37.567657,126.979344,MOVE
5,V01,2026-09-25 09:25:00,37.568824,126.980657,MOVE
6,V01,2026-09-25 09:30:00,37.570005,126.981962,MOVE
7,V01,2026-09-25 09:35:00,37.569966,126.981989,STOP
8,V01,2026-09-25 09:40:00,37.569980,126.982006,STOP
9,V01,2026-09-25 09:45:00,37.569982,126.981972,STOP


## 3. Spatial Grid Assignment
GPS 포인트를 약 200m 수준의 격자로 집계한다. 격자 크기는 **PoC용 초기 설정값**이며, 실제 데이터 적용 시 도시밀도·GPS 오차·광고 단위에 맞춰 조정한다.

In [ ]:
gps["grid_lat"] = np.floor(gps["latitude"] / GRID_SIZE) * GRID_SIZE
gps["grid_lon"] = np.floor(gps["longitude"] / GRID_SIZE) * GRID_SIZE
gps["grid_id"] = (
    gps["grid_lat"].round(3).astype(str) + "_" + gps["grid_lon"].round(3).astype(str)
)

print("분석 격자 수:", gps["grid_id"].nunique())
display(gps[["vehicle_id", "timestamp", "latitude", "longitude", "grid_id"]].head())

분석 격자 수: 19


,vehicle_id,timestamp,latitude,longitude,grid_id
0,V01,2026-09-25 09:00:00,37.566510,126.977997,37.566_126.976
1,V01,2026-09-25 09:05:00,37.566513,126.978030,37.566_126.978
2,V01,2026-09-25 09:10:00,37.566495,126.977995,37.566_126.976
3,V01,2026-09-25 09:15:00,37.566532,126.978015,37.566_126.978
4,V01,2026-09-25 09:20:00,37.567657,126.979344,37.566_126.978


## 4. Stop Detection from GPS Speed
생성 단계의 `scenario_state` 라벨을 분석에 직접 사용하지 않고, **GPS 좌표와 시간차만으로 속도를 계산**하여 정차 후보를 탐지한다.

- Haversine 거리 계산
- 속도 ≤ 1 km/h: 정차 후보
- 연속 정차 후보를 하나의 episode로 묶음
- 체류시간 ≥ 10분인 episode만 유효 정차로 채택

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

gps["prev_lat"] = gps.groupby("vehicle_id")["latitude"].shift(1)
gps["prev_lon"] = gps.groupby("vehicle_id")["longitude"].shift(1)
gps["prev_time"] = gps.groupby("vehicle_id")["timestamp"].shift(1)

gps["distance_km"] = gps.apply(
    lambda r: haversine_km(r["prev_lat"], r["prev_lon"], r["latitude"], r["longitude"]), axis=1
)
gps["time_diff_h"] = (gps["timestamp"] - gps["prev_time"]).dt.total_seconds() / 3600
gps["speed_kmh"] = (gps["distance_km"] / gps["time_diff_h"]).replace([np.inf, -np.inf], np.nan)

# 차량별 첫 포인트는 정차판정에서 제외
gps["stop_candidate"] = gps["speed_kmh"].le(STOP_SPEED_KMH).fillna(False)
first_idx = gps.groupby("vehicle_id").head(1).index
gps.loc[first_idx, "stop_candidate"] = False

# 연속 상태를 episode로 묶기
gps["stop_change"] = gps.groupby("vehicle_id")["stop_candidate"].transform(lambda x: x.ne(x.shift()))
gps["episode_id"] = gps.groupby("vehicle_id")["stop_change"].cumsum()

display(gps[["vehicle_id", "timestamp", "speed_kmh", "stop_candidate", "grid_id"]].head(15))

,vehicle_id,timestamp,speed_kmh,stop_candidate,grid_id
0,V01,2026-09-25 09:00:00,NaN,False,37.566_126.976
1,V01,2026-09-25 09:05:00,0.035372,True,37.566_126.978
2,V01,2026-09-25 09:10:00,0.043993,True,37.566_126.976
3,V01,2026-09-25 09:15:00,0.052827,True,37.566_126.978
4,V01,2026-09-25 09:20:00,2.057048,False,37.566_126.978
5,V01,2026-09-25 09:25:00,2.086340,False,37.568_126.98
6,V01,2026-09-25 09:30:00,2.094157,False,37.57_126.98
7,V01,2026-09-25 09:35:00,0.059764,True,37.568_126.98
8,V01,2026-09-25 09:40:00,0.026550,True,37.568_126.982
9,V01,2026-09-25 09:45:00,0.036627,True,37.568_126.98


In [ ]:
stop_points = gps[gps["stop_candidate"]].copy()

stop_events = (
    stop_points
    .groupby(["vehicle_id", "episode_id"])
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        grid_id=("grid_id", lambda x: x.mode().iloc[0]),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        gps_points=("timestamp", "count"),
    )
    .reset_index()
)

# 샘플 간격을 포함한 체류시간 추정
stop_events["dwell_minutes"] = stop_events["gps_points"] * SAMPLE_INTERVAL_MIN
valid_stops = stop_events[stop_events["dwell_minutes"] >= MIN_DWELL_MIN].copy()

print(f"정차 episode: {len(stop_events)} | 유효 정차 episode: {len(valid_stops)}")
display(valid_stops)

정차 episode: 11 | 유효 정차 episode: 11


,vehicle_id,episode_id,start_time,end_time,grid_id,latitude,longitude,gps_points,dwell_minutes
0,V01,2,2026-09-25 09:05:00,2026-09-25 09:15:00,37.566_126.978,37.566513,126.978014,3,15
1,V01,4,2026-09-25 09:35:00,2026-09-25 09:55:00,37.568_126.98,37.569992,126.981987,5,25
2,V01,6,2026-09-25 10:15:00,2026-09-25 10:35:00,37.566_126.976,37.566496,126.977991,5,25
3,V01,8,2026-09-25 10:55:00,2026-09-25 11:00:00,37.562_126.982,37.562499,126.983993,2,10
4,V02,2,2026-09-25 09:15:00,2026-09-25 09:25:00,37.568_126.98,37.569997,126.982008,3,15
5,V02,4,2026-09-25 09:45:00,2026-09-25 10:05:00,37.562_126.984,37.562506,126.984019,5,25
6,V02,6,2026-09-25 10:25:00,2026-09-25 10:40:00,37.568_126.98,37.570007,126.981998,4,20
7,V02,8,2026-09-25 11:00:00,2026-09-25 11:05:00,37.566_126.97,37.567999,126.971988,2,10
8,V03,2,2026-09-25 09:25:00,2026-09-25 09:30:00,37.566_126.976,37.566482,126.977988,2,10
9,V03,4,2026-09-25 09:50:00,2026-09-25 10:05:00,37.566_126.97,37.567990,126.972008,4,20


## 5. Mobility Feature Engineering
Transit OOH 관점에서 배송차량의 공간 활동을 다음 세 지표로 해석한다.

- **Visit Frequency**: 해당 공간에 광고매체가 반복적으로 등장하는 정도
- **Dwell Time**: 해당 공간에서 머무르는 잠재 노출시간
- **Repeat Visit**: 동일 차량의 반복 방문에 따른 반복 노출 가능성

In [ ]:
visit_features = (
    valid_stops.groupby("grid_id")
    .agg(
        visit_frequency=("episode_id", "count"),
        unique_vehicles=("vehicle_id", "nunique"),
    )
    .reset_index()
)

dwell_features = (
    valid_stops.groupby("grid_id")
    .agg(
        total_dwell_minutes=("dwell_minutes", "sum"),
        avg_dwell_minutes=("dwell_minutes", "mean"),
    )
    .reset_index()
)

vehicle_grid_visits = (
    valid_stops.groupby(["vehicle_id", "grid_id"]).size().reset_index(name="visit_count")
)
vehicle_grid_visits["repeat_visit_count"] = (vehicle_grid_visits["visit_count"] - 1).clip(lower=0)

repeat_features = (
    vehicle_grid_visits.groupby("grid_id")
    .agg(repeat_visits=("repeat_visit_count", "sum"))
    .reset_index()
)

mobility = (
    visit_features.merge(dwell_features, on="grid_id", how="outer")
    .merge(repeat_features, on="grid_id", how="outer")
    .fillna(0)
)

display(mobility)

,grid_id,visit_frequency,unique_vehicles,total_dwell_minutes,avg_dwell_minutes,repeat_visits
0,37.562_126.982,1,1,10,10.000000,0
1,37.562_126.984,1,1,25,25.000000,0
2,37.566_126.97,2,2,30,15.000000,0
3,37.566_126.976,3,2,50,16.666667,1
4,37.566_126.978,1,1,15,15.000000,0
5,37.568_126.98,3,2,60,20.000000,1


## 6. Normalization & Mobility Score
서로 단위가 다른 특징을 0~1로 정규화하고 초기 가중치로 Mobility Score를 구성한다.

In [ ]:
def min_max_normalize(series):
    s = pd.Series(series, index=series.index, dtype=float)
    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

mobility["visit_score"] = min_max_normalize(mobility["visit_frequency"])
mobility["dwell_score"] = min_max_normalize(mobility["total_dwell_minutes"])
mobility["repeat_score"] = min_max_normalize(mobility["repeat_visits"])

mobility["mobility_score"] = (
    W_VISIT * mobility["visit_score"]
    + W_DWELL * mobility["dwell_score"]
    + W_REPEAT * mobility["repeat_score"]
)

display(mobility.sort_values("mobility_score", ascending=False))

,grid_id,visit_frequency,unique_vehicles,total_dwell_minutes,avg_dwell_minutes,repeat_visits,visit_score,dwell_score,repeat_score,mobility_score
5,37.568_126.98,3,2,60,20.000000,1,1.0,1.0,1.0,1.00
3,37.566_126.976,3,2,50,16.666667,1,1.0,0.8,1.0,0.92
2,37.566_126.97,2,2,30,15.000000,0,0.5,0.4,0.0,0.36
1,37.562_126.984,1,1,25,25.000000,0,0.0,0.3,0.0,0.12
4,37.566_126.978,1,1,15,15.000000,0,0.0,0.1,0.0,0.04
0,37.562_126.982,1,1,10,10.000000,0,0.0,0.0,0.0,0.00


## 7. Simulated Spatial Context Integration
실제 유동인구·POI 데이터 확보 전 단계이므로, 외부 공간정보 결합 구조를 확인하기 위한 **시나리오형 공간 데이터**를 사용한다. 아래 값은 실측치가 아니며 광고효과 검증에 사용하지 않는다.

In [ ]:
# 격자 중심좌표
grid_coords = (
    gps.groupby("grid_id")
    .agg(grid_lat=("grid_lat", "first"), grid_lon=("grid_lon", "first"))
    .reset_index()
)

result = mobility.merge(grid_coords, on="grid_id", how="left")
result = result.sort_values(["grid_lat", "grid_lon"]).reset_index(drop=True)

# 외부 공간데이터 결합 구조를 보여주기 위한 고정 시나리오 값
scenario_population = [4200, 1800, 3500, 900, 2800, 1500, 3200, 2200]
scenario_poi =        [  48,   16,   35,   8,   27,   12,   31,   20]

n = len(result)
result["floating_population"] = [scenario_population[i % len(scenario_population)] for i in range(n)]
result["poi_count"] = [scenario_poi[i % len(scenario_poi)] for i in range(n)]

result["population_score"] = min_max_normalize(result["floating_population"])
result["poi_score"] = min_max_normalize(result["poi_count"])
result["spatial_score"] = 0.7 * result["population_score"] + 0.3 * result["poi_score"]

display(result[["grid_id", "mobility_score", "floating_population", "poi_count", "spatial_score"]])

,grid_id,mobility_score,floating_population,poi_count,spatial_score
0,37.562_126.982,0.00,4200,48,1.000000
1,37.562_126.984,0.12,1800,16,0.250909
2,37.566_126.97,0.36,3500,35,0.754015
3,37.566_126.976,0.92,900,8,0.000000
4,37.566_126.978,0.04,2800,27,0.545530
5,37.568_126.98,1.00,1500,12,0.157273


## 8. Proposed Location Value
제안모델은 **이동특성(Mobility)** 과 **정적 공간특성(Spatial)** 을 함께 반영한다.



$Location\ Value = 0.6 	imes Mobility\ Score + 0.4 	imes Spatial\ Score$

현재 가중치는 PoC 초기값이며, 아래 민감도 분석으로 특정 가중치에 결과가 과도하게 종속되는지 확인한다.

In [ ]:
result["location_value"] = (
    ALPHA_MOBILITY * result["mobility_score"]
    + BETA_SPATIAL * result["spatial_score"]
)

result["proposed_rank"] = result["location_value"].rank(method="first", ascending=False).astype(int)
result = result.sort_values("proposed_rank").reset_index(drop=True)

display(result[[
    "proposed_rank", "grid_id", "visit_frequency", "total_dwell_minutes",
    "repeat_visits", "mobility_score", "spatial_score", "location_value"
]])

,proposed_rank,grid_id,visit_frequency,total_dwell_minutes,repeat_visits,mobility_score,spatial_score,location_value
0,1,37.568_126.98,3,60,1,1.00,0.157273,0.662909
1,2,37.566_126.976,3,50,1,0.92,0.000000,0.552000
2,3,37.566_126.97,2,30,0,0.36,0.754015,0.517606
3,4,37.562_126.982,1,10,0,0.00,1.000000,0.400000
4,5,37.566_126.978,1,15,0,0.04,0.545530,0.242212
5,6,37.562_126.984,1,25,0,0.12,0.250909,0.172364


## 9. Static-only vs Proposed Model
**Static-only**는 유동인구·POI와 같은 정적 공간정보만 사용하고, **Proposed**는 배송차량의 방문·체류·재방문 정보를 추가한다.

이 비교는 “추가 설명력이 입증되었다”는 의미가 아니다. 대신 **궤적 기반 정보가 추가될 때 후보지 평가 순위가 어떻게 달라질 수 있는지**를 PoC 수준에서 확인한다.

In [ ]:
comparison = result.copy()
comparison["static_rank"] = comparison["spatial_score"].rank(method="first", ascending=False).astype(int)
comparison["rank_change"] = comparison["static_rank"] - comparison["proposed_rank"]

comparison_view = comparison[[
    "grid_id", "spatial_score", "static_rank", "mobility_score",
    "location_value", "proposed_rank", "rank_change"
]].sort_values("proposed_rank")

print("rank_change > 0 : 궤적 정보를 추가했을 때 순위 상승")
print("rank_change < 0 : 궤적 정보를 추가했을 때 순위 하락")
display(comparison_view)

rank_change > 0 : 궤적 정보를 추가했을 때 순위 상승
rank_change < 0 : 궤적 정보를 추가했을 때 순위 하락


,grid_id,spatial_score,static_rank,mobility_score,location_value,proposed_rank,rank_change
0,37.568_126.98,0.157273,5,1.00,0.662909,1,4
1,37.566_126.976,0.000000,6,0.92,0.552000,2,4
2,37.566_126.97,0.754015,2,0.36,0.517606,3,-1
3,37.562_126.982,1.000000,1,0.00,0.400000,4,-3
4,37.566_126.978,0.545530,3,0.04,0.242212,5,-2
5,37.562_126.984,0.250909,4,0.12,0.172364,6,-2


## 10. Weight Sensitivity Analysis
Mobility:Spatial 비율을 바꾸어도 상위 후보가 크게 뒤집히는지 확인한다. 이는 가중치의 정답을 찾는 검증이 아니라 **초기 가중치에 대한 결과 민감도 확인**이다.

In [ ]:
scenarios = {
    "M40_S60": 0.4,
    "M50_S50": 0.5,
    "M60_S40_BASE": 0.6,
    "M70_S30": 0.7,
}

sensitivity = result[["grid_id", "mobility_score", "spatial_score"]].copy()

for name, alpha in scenarios.items():
    score_col = f"score_{name}"
    rank_col = f"rank_{name}"
    sensitivity[score_col] = alpha * sensitivity["mobility_score"] + (1-alpha) * sensitivity["spatial_score"]
    sensitivity[rank_col] = sensitivity[score_col].rank(method="first", ascending=False).astype(int)

rank_cols = [c for c in sensitivity.columns if c.startswith("rank_")]
sensitivity["rank_range"] = sensitivity[rank_cols].max(axis=1) - sensitivity[rank_cols].min(axis=1)

base_rank = sensitivity["rank_M60_S40_BASE"]
stability = []
for name in scenarios:
    r = sensitivity[f"rank_{name}"]
    stability.append({
        "scenario": name,
        "spearman_vs_base": r.corr(base_rank, method="spearman"),
        "top_grid": sensitivity.loc[r.idxmin(), "grid_id"],
    })

stability_df = pd.DataFrame(stability)

print("가중치별 순위")
display(sensitivity[["grid_id"] + rank_cols + ["rank_range"]].sort_values("rank_M60_S40_BASE"))
print("기준 시나리오(M60:S40) 대비 순위 안정성")
display(stability_df)

가중치별 순위


,grid_id,rank_M40_S60,rank_M50_S50,rank_M60_S40_BASE,rank_M70_S30,rank_range
0,37.568_126.98,3,1,1,1,2
1,37.566_126.976,4,4,2,2,2
2,37.566_126.97,2,2,3,3,1
3,37.562_126.982,1,3,4,4,3
4,37.566_126.978,5,5,5,5,0
5,37.562_126.984,6,6,6,6,0


기준 시나리오(M60:S40) 대비 순위 안정성


,scenario,spearman_vs_base,top_grid
0,M40_S60,0.485714,37.562_126.982
1,M50_S50,0.828571,37.568_126.98
2,M60_S40_BASE,1.000000,37.568_126.98
3,M70_S30,1.000000,37.568_126.98


## 11. Transit OOH Interpretation
본 PoC에서 지표는 다음과 같이 해석한다.

| 분석 변수 | Transit OOH 해석 |
|---|---|
| 방문빈도 | 광고매체가 해당 공간에 등장하는 빈도 |
| 체류시간 | 해당 공간에서의 잠재 노출시간 |
| 재방문 | 동일 지역 반복 노출 가능성 |
| 유동인구 | 잠재 광고 수용자 규모의 공간 proxy |
| POI | 상업·업무·생활 활동성이 존재하는 공간 proxy |

따라서 제안모델은 **배송차량이 어디를 자주·오래·반복해서 방문하는지**와 **그 공간 자체의 외부 특성**을 결합해 상대적인 OOH 후보 위치를 제시한다.

## 12. GIS Dashboard
차량 궤적, Location Value 격자, Top 3 후보를 하나의 GIS 화면에 통합한다. 지도 내 점수는 **상대적 PoC 지표**이다.

In [ ]:
dashboard_df = comparison.sort_values("proposed_rank").copy()
top3 = dashboard_df.head(3)

center_lat = gps["latitude"].mean()
center_lon = gps["longitude"].mean()

dashboard_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles=None,
    control_scale=True,
)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors, Tiles style by HOT",
    name="Base Map",
).add_to(dashboard_map)

trajectory_layer = folium.FeatureGroup(name="Vehicle Trajectory", show=True)
for vehicle in gps["vehicle_id"].unique():
    v = gps[gps["vehicle_id"] == vehicle].sort_values("timestamp")
    coords = v[["latitude", "longitude"]].values.tolist()
    folium.PolyLine(coords, weight=3, opacity=0.65, tooltip=f"Vehicle: {vehicle}").add_to(trajectory_layer)
trajectory_layer.add_to(dashboard_map)

lv_min, lv_max = dashboard_df["location_value"].min(), dashboard_df["location_value"].max()
if lv_min == lv_max:
    lv_max = lv_min + 0.001
colormap = cm.linear.YlOrRd_09.scale(lv_min, lv_max)
colormap.caption = "Relative Transit OOH Location Value"

grid_layer = folium.FeatureGroup(name="Location Value Grid", show=True)
for _, row in dashboard_df.iterrows():
    popup_html = f"""
    <div style='font-family:Arial;width:245px;line-height:1.6'>
      <b>Candidate Rank #{int(row['proposed_rank'])}</b><hr style='margin:6px 0'>
      Grid ID: {row['grid_id']}<br>
      Mobility Score: {row['mobility_score']:.3f}<br>
      Spatial Score: {row['spatial_score']:.3f}<br>
      <b>Location Value: {row['location_value']:.3f}</b><br>
      Static-only Rank: #{int(row['static_rank'])}<br>
      Proposed Rank: #{int(row['proposed_rank'])}<br>
      Rank Change: {int(row['rank_change']):+d}
    </div>
    """
    folium.Rectangle(
        bounds=[[row['grid_lat'], row['grid_lon']],
                [row['grid_lat'] + GRID_SIZE, row['grid_lon'] + GRID_SIZE]],
        color="#FFFFFF", weight=1.2, fill=True,
        fill_color=colormap(row["location_value"]), fill_opacity=0.72,
        tooltip=f"Rank #{int(row['proposed_rank'])} | LV {row['location_value']:.3f}",
        popup=folium.Popup(popup_html, max_width=290),
    ).add_to(grid_layer)
grid_layer.add_to(dashboard_map)
colormap.add_to(dashboard_map)

candidate_layer = folium.FeatureGroup(name="Top 3 OOH Candidates", show=True)
for _, row in top3.iterrows():
    lat = row["grid_lat"] + GRID_SIZE / 2
    lon = row["grid_lon"] + GRID_SIZE / 2
    folium.Marker(
        [lat, lon],
        icon=folium.DivIcon(html=f"""<div style='background:white;border:3px solid #222;border-radius:50%;width:34px;height:34px;text-align:center;line-height:29px;font-weight:bold;font-size:14px;box-shadow:0 2px 7px rgba(0,0,0,.35)'>{int(row['proposed_rank'])}</div>"""),
        tooltip=f"OOH Candidate #{int(row['proposed_rank'])} | LV {row['location_value']:.3f}",
    ).add_to(candidate_layer)
candidate_layer.add_to(dashboard_map)

header = f"""
<div style='position:fixed;top:15px;left:50%;transform:translateX(-50%);width:78%;z-index:9999;font-family:Arial'>
<div style='background:rgba(255,255,255,.96);border-radius:12px;padding:15px 20px;box-shadow:0 3px 16px rgba(0,0,0,.18)'>
<div style='display:flex;justify-content:space-between;align-items:center'>
<div><div style='font-size:20px;font-weight:700'>Transit OOH Location Value Analysis</div>
<div style='font-size:11px;color:#777'>Last-mile Vehicle Trajectory + Spatial Context</div></div>
<div style='font-size:11px;font-weight:bold;background:#eee;padding:5px 10px;border-radius:12px'>PoC · Simulated Data</div>
</div><hr style='border:none;border-top:1px solid #e8e8e8;margin:10px 0'>
<div style='display:flex;justify-content:space-around;text-align:center'>
<div><div style='font-size:10px;color:#888'>GRIDS</div><b>{len(dashboard_df)}</b></div>
<div><div style='font-size:10px;color:#888'>TOP LV</div><b>{dashboard_df['location_value'].max():.3f}</b></div>
<div><div style='font-size:10px;color:#888'>AVG MOBILITY</div><b>{dashboard_df['mobility_score'].mean():.3f}</b></div>
<div><div style='font-size:10px;color:#888'>AVG SPATIAL</div><b>{dashboard_df['spatial_score'].mean():.3f}</b></div>
</div></div></div>
"""
dashboard_map.get_root().html.add_child(Element(header))

folium.LayerControl(position="bottomleft", collapsed=False).add_to(dashboard_map)
dashboard_map

NameError: name 'comparison' is not defined

## 13. Export Results
발표 및 후속 분석을 위해 결과 테이블·민감도 분석·GIS 대시보드를 저장한다.

In [ ]:
result_export = comparison.sort_values("proposed_rank").copy()
result_export.to_csv("transit_ooh_location_value_results.csv", index=False, encoding="utf-8-sig")
sensitivity.to_csv("transit_ooh_sensitivity_analysis.csv", index=False, encoding="utf-8-sig")
dashboard_map.save("transit_ooh_dashboard.html")

print("저장 완료")
print("- transit_ooh_location_value_results.csv")
print("- transit_ooh_sensitivity_analysis.csv")
print("- transit_ooh_dashboard.html")

NameError: name 'comparison' is not defined

## 14. Interpretation & Next Step
### 이번 PoC에서 확인한 것
1. 배송차량 GPS로부터 정차·방문·체류·재방문 특징을 추출할 수 있다.
2. 이동특성과 외부 공간특성을 결합하여 격자별 상대적 Location Value를 산출할 수 있다.
3. 정적 공간지표만 사용한 후보지 순위와 궤적 정보를 추가한 후보지 순위가 달라질 수 있음을 확인할 수 있다.
4. 결과를 GIS 대시보드로 시각화하여 후보 위치를 직관적으로 비교할 수 있다.

### 아직 검증하지 않은 것
- 실제 광고 노출량·성과에 대한 설명력
- 실측 유동인구·POI 기반의 외부타당도
- 최적 가중치의 통계적 추정
- 다른 지역·기간에서의 일반화 가능성

### 후속 단계
**실제 배송 GPS + 실제 유동인구/POI + 광고성과 proxy**를 확보한 뒤, `Static-only Model`과 `Static + Mobility Model`의 설명력을 비교하여 궤적 기반 지표의 증분 설명력을 검증한다.